# GeoQuant AI — Hello World
Objectif : Afficher le prix du S&P500 **à côté** d'une headline de news pour la même date.

In [ ]:
import sys
from pathlib import Path

# Ajoute src/ au path pour pouvoir importer nos modules
sys.path.insert(0, str(Path('..') / 'src'))

## 1. Chargement des prix (S&P500)

In [ ]:
import yfinance as yf
import pandas as pd

prices = yf.download('^GSPC', start='2017-01-01', end='2018-12-31', progress=False, auto_adjust=False)
prices.index = pd.to_datetime(prices.index)
prices = prices[['Close']].rename(columns={'Close': 'SP500_Close'})
prices.index.name = 'date'
print(f'Prix chargés : {len(prices)} jours')
prices.head()

## 2. Chargement des news

> **Avant d'exécuter cette cellule**, place ton CSV Kaggle dans `data/raw/` et ajuste `CSV_FILE` et `DATASET` ci-dessous.

In [ ]:
from news_loader import NewsLoader

base_dir = Path('..')

DATASET  = 'financial_news'          # ou 'all_the_news'
CSV_FILE = 'combined_news_data.csv'  # nom du fichier dans data/raw/

loader = NewsLoader(
    csv_path  = base_dir / 'data' / 'raw' / CSV_FILE,
    dataset   = DATASET,
    start_date= '2017-01-01',
    end_date  = '2018-12-31',
    base_dir  = base_dir,
)

raw_df  = loader.load_raw()
news_df = loader.process(raw_df)
news_df.head()

## 3. Jointure Prix + News (Hello World)

In [ ]:
# Garder uniquement la première headline par jour
news_daily = news_df.groupby('date').first().reset_index()
news_daily = news_daily.set_index('date')

# Jointure sur la date
merged = prices.join(news_daily[['headline', 'source']], how='inner')

print(f'Jours avec prix ET news : {len(merged)}')
merged.head(10)